# Baseline

In [1]:
import sys
sys.path.append('../')

import pandas as pd, numpy as np, ast, wfdb
from src.preprocessing.dataset import ECGDataset
from src.models.baseline_cnn import BaselineCNN
from src.training.train import train_model
from src.utils.config import CFG

In [2]:
# Load data
path = CFG['data']['path']
Y = pd.read_csv(path + 'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

In [3]:
from src.preprocessing.label_utils import load_all_labels

# Build Y with 'superclass' (list) and 'label_vec' (multi-hot) columns
# ECGDataset loads signals on-the-fly — no need to pre-load the full array
Y = load_all_labels(path + 'ptbxl_database.csv', path + 'scp_statements.csv')

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)


In [4]:
# Split, ECGDataset takes a dataframe slice + the data path; loads signals on-the-fly
train_ds = ECGDataset(Y[Y.strat_fold < 9],  path)
val_ds   = ECGDataset(Y[Y.strat_fold == 9], path)

In [5]:
from src.models.dummy_classifier import DummyECGClassifier

# Build test split here so dummy can evaluate all three splits
test_ds = ECGDataset(Y[Y.strat_fold == 10], CFG['data']['path'])

dummy = DummyECGClassifier()
dummy.fit(train_ds)

print("\nValidation:")
val_metrics = dummy.evaluate(val_ds)
for cls, auc in val_metrics['per_class_auc'].items():
    print(f"  {cls:5s}: {auc:.4f}")
print(f"  AUC macro: {val_metrics['auc_macro']:.4f}  F1 macro: {val_metrics['f1_macro']:.4f}")

print("\nTest:")
test_metrics = dummy.evaluate(test_ds)
for cls, auc in test_metrics['per_class_auc'].items():
    print(f"  {cls:5s}: {auc:.4f}")
print(f"  AUC macro: {test_metrics['auc_macro']:.4f}  F1 macro: {test_metrics['f1_macro']:.4f}")

dummy.save_results(test_metrics, 'dummy_metrics.json')

Dummy classifier fitted (strategy='prior') on class frequencies:
  NORM : 0.445 (44.5%)
  MI   : 0.256 (25.6%)
  STTC : 0.245 (24.5%)
  CD   : 0.229 (22.9%)
  HYP  : 0.124 (12.4%)

Validation:
  NORM : 0.5000
  MI   : 0.5000
  STTC : 0.5000
  CD   : 0.5000
  HYP  : 0.5000
  AUC macro: 0.5000  F1 macro: 0.0000

Test:
  NORM : 0.5000
  MI   : 0.5000
  STTC : 0.5000
  CD   : 0.5000
  HYP  : 0.5000
  AUC macro: 0.5000  F1 macro: 0.0000
Saved -> D:\GitHub\biosignal-xai\results/dummy_metrics.json


In [ ]:
# Train baseline
model = BaselineCNN(num_classes=5)
train_model(model, train_ds, val_ds, epochs=CFG['training']['epochs'],
            save_dir=CFG['paths']['results'] + 'baseline_cnn')

Training on: cuda


## Evaluation on the Test Set

Fold 10 is the held-out test set, never seen during training or validation. We:
1. Rebuild the label dataframe with `label_vec` (multi-hot vectors) using `load_all_labels`
2. Load the best checkpoint saved during training (`../results/baseline_cnn/checkpoint.pt`)
3. Run inference on fold 10 and collect logits
4. Compute **macro AUC** and **macro F1**, plus per-class AUC — the metrics used in PTB-XL benchmark papers
5. Save results to `../results/baseline_cnn_metrics.json` for comparison with the Transformer model later

In [6]:
from src.preprocessing.label_utils import load_all_labels
from src.utils.metrics import compute_metrics, print_metrics

# Build dataframe with label_vec (multi-hot) — required by ECGDataset
Y_eval = load_all_labels(path + 'ptbxl_database.csv', path + 'scp_statements.csv')

test_df = Y_eval[Y_eval.strat_fold == 10]
test_ds = ECGDataset(test_df, path)
print(f"Test records: {len(test_df)}  |  Test windows: {len(test_ds)}")

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Test records: 2158  |  Test windows: 15106


In [ ]:
import torch
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoint
model = BaselineCNN(num_classes=5)
model.load_state_dict(torch.load(
    CFG['paths']['results'] + 'baseline_cnn/checkpoint.pt',
    map_location=device, weights_only=True
))
model = model.to(device)
model.eval()

test_loader = DataLoader(test_ds, batch_size=CFG['training']['batch_size'])

all_logits, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        all_logits.append(model(x.to(device)).cpu())
        all_labels.append(y)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

metrics = compute_metrics(all_logits, all_labels)
print("Baseline CNN — Test Set Results")
print("=" * 35)
print_metrics(metrics)

In [ ]:
import json, os

save_dir = CFG['paths']['results'] + 'baseline_cnn'
os.makedirs(save_dir, exist_ok=True)

results = {
    'model': 'BaselineCNN',
    'auc_macro': round(metrics['auc_macro'], 4),
    'f1_macro':  round(metrics['f1_macro'],  4),
    'per_class_auc': {k: round(v, 4) for k, v in metrics['per_class_auc'].items()}
}

out_path = os.path.join(save_dir, 'baseline_cnn_metrics.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved -> {out_path}")